# 🌿 Plant Disease Detection using Deep Learning

**Project Title:** Plant Disease Detection using Transfer Learning (MobileNetV2) on the PlantVillage Dataset

**Goal:** Pattey ki image se plant ki disease (ya healthy) classify karna.

**Dataset:** PlantVillage (~54,000 images, 38 classes) - TensorFlow Datasets se automatically download hota hai.

**Model:** MobileNetV2 (ImageNet pretrained) + custom classification head.

> Tip: Google Colab me run karein aur Runtime > Change runtime type > **GPU** select karein.

## 1. Libraries Import

In [ ]:
!pip install -q tensorflow tensorflow-datasets scikit-learn seaborn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Dataset Load (Train / Validation / Test = 80 / 10 / 10)

In [ ]:
(train_ds, val_ds, test_ds), info = tfds.load(
    'plant_village',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True
)

class_names = info.features['label'].names
num_classes = len(class_names)
print('Number of classes:', num_classes)
print(class_names)

## 3. Sample Images Dekhein

In [ ]:
plt.figure(figsize=(12, 8))
for i, (img, label) in enumerate(train_ds.take(8)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(img.numpy())
    plt.title(class_names[label.numpy()].replace('___', '\n'), fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Preprocessing & Data Pipeline

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def prepare(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)   # scales pixels to [-1, 1]
    return image, label

train_data = train_ds.map(prepare, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_data   = val_ds.map(prepare, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_data  = test_ds.map(prepare, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 5. Model Build (Transfer Learning + Data Augmentation)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False   # pehle sirf head train hoga

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 6. Training (Stage 1: Head Training)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2)
]

history = model.fit(train_data, validation_data=val_data, epochs=5, callbacks=callbacks)

## 7. Fine-Tuning (Stage 2: Last layers unfreeze)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:   # sirf last 30 layers train honge
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(train_data, validation_data=val_data, epochs=5, callbacks=callbacks)

## 8. Accuracy & Loss Graphs

In [ ]:
acc  = history.history['accuracy'] + history_ft.history['accuracy']
val_acc = history.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history.history['loss'] + history_ft.history['loss']
val_loss = history.history['val_loss'] + history_ft.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train'); plt.plot(val_acc, label='Validation')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train'); plt.plot(val_loss, label='Validation')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.show()

## 9. Test Set Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print(f'Test Accuracy: {test_acc*100:.2f}%')

y_true, y_pred = [], []
for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

## 10. Prediction on Test Images

In [ ]:
plt.figure(figsize=(14, 8))
for i, (img, label) in enumerate(test_ds.take(8)):
    x = prepare(img, label)[0]
    pred = model.predict(tf.expand_dims(x, 0), verbose=0)[0]
    pred_class = class_names[np.argmax(pred)]
    conf = np.max(pred) * 100
    actual = class_names[label.numpy()]
    plt.subplot(2, 4, i + 1)
    plt.imshow(img.numpy())
    color = 'green' if pred_class == actual else 'red'
    plt.title(f'Pred: {pred_class.replace("___", " - ")}\n({conf:.1f}%)\nActual: {actual.replace("___", " - ")}', fontsize=7, color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 11. Apni Khud Ki Image Par Predict Karein

In [ ]:
from google.colab import files   # sirf Colab me chalega
from tensorflow.keras.preprocessing import image

uploaded = files.upload()
for fname in uploaded.keys():
    img = image.load_img(fname, target_size=(IMG_SIZE, IMG_SIZE))
    arr = preprocess_input(np.expand_dims(image.img_to_array(img), 0))
    pred = model.predict(arr, verbose=0)[0]
    plt.imshow(img); plt.axis('off')
    plt.title(f'{class_names[np.argmax(pred)]} ({np.max(pred)*100:.1f}%)')
    plt.show()

## 12. Model Save

In [ ]:
model.save('plant_disease_model.keras')
print('Model saved: plant_disease_model.keras')

## ✅ Conclusion

- MobileNetV2 transfer learning se PlantVillage dataset par high accuracy (~95%+) mil sakti hai.
- Future scope: Streamlit/Flask web app, mobile app (TFLite), real-field images par training, Grad-CAM se explainability.